In [1]:
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source

In [2]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })


In [26]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optional)
)

In [4]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 107.11 MB (60.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.64s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 7626007.27/s
Memory usage: 109.49 MB (60.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.43s | Rate: 0.00/s
books: 300  | Time: 1.79s | Rate: 167.50/s
Memory usage: 109.99 MB (60.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.62s | Rate: 0.00/s
books: 500  | Time: 2.99s | Rate: 167.46/s
Memory usage: 110.49 MB (60.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 6.18s | Rate: 0.

In [25]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

{'started_at': DateTime(2026, 2, 21, 8, 34, 45, 218678, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 2, 21, 8, 35, 14, 825940, tzinfo=Timezone('UTC')), 'schema_name': 'rest_api', 'job_metrics': {'books.8bb7948151.typed-jsonl.gz': DataWriterMetrics(file_path='/home/freeman/.dlt/pipelines/ol_demo/normalize/91a0b92c7cd63dd7/1771662885.2109208/new_jobs/books.8bb7948151.0.typed-jsonl.gz', items_count=3756, file_size=1284545, created=1771662886.8558173, last_modified=1771662914.2566319)}, 'table_metrics': {'books': DataWriterMetrics(file_path='', items_count=3756, file_size=1284545, created=1771662886.8558173, last_modified=1771662914.2566319)}, 'resource_metrics': {'books': DataWriterAndCustomMetrics(file_path='', items_count=3756, file_size=1284545, created=1771662886.8558173, last_modified=1771662914.2566319)}, 'dag': [('books', 'books')], 'hints': {'books': {'write_disposition': 'replace', 'primary_key': 'key'}}}
Resources: ['books']
Tables: ['books']
Load ID: 1771662885.210920

In [7]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1771662885.2109208 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 119.82 MB (60.80%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1771662885.2109208 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 119.82 MB (60.80%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1771662885.2109208 -------------------
Files: 9/1 (900.0%) | Time: 1.00s | Rate: 9.03/s
Items: 22900  | Time: 1.00s | Rate: 23001.87/s
Memory usage: 125.02 MB (60.80%) | CPU usage: 0.00%



In [8]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")


Load ID: 1771662885.2109208

Tables created/updated:
  - books: 3756 rows
  - books__author_key: 4600 rows
  - books__author_name: 4600 rows
  - books__ia: 3406 rows
  - books__ia_collection: 2670 rows
  - books__language: 3742 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 60 rows
  - books__id_project_gutenberg: 54 rows


In [9]:
# Display schema 
pipeline.default_schema

<dlt.Schema(name='rest_api', version=2, tables=['_dlt_version', '_dlt_loads', 'books', '_dlt_pipeline_state', 'books__author_key', 'books__author_name', 'books__ia', 'books__ia_collection', 'books__language', 'books__id_standard_ebooks', 'books__id_librivox', 'books__id_project_gutenberg'], version_hash='ZJIabaQJ9DAYgsR04wEVeXOgU80roBUfdvrR2YoBEyU=')>

In [10]:
load_info = pipeline.load()

--------------------- Load rest_api in 1771662885.2109208 ----------------------
Jobs: 0/9 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 150.46 MB (61.00%) | CPU usage: 0.00%

--------------------- Load rest_api in 1771662885.2109208 ----------------------
Jobs: 2/9 (22.2%) | Time: 1.89s | Rate: 1.06/s
Memory usage: 223.64 MB (62.20%) | CPU usage: 0.00%

--------------------- Load rest_api in 1771662885.2109208 ----------------------
Jobs: 8/9 (88.9%) | Time: 3.34s | Rate: 2.40/s
Memory usage: 201.88 MB (62.10%) | CPU usage: 0.00%

--------------------- Load rest_api in 1771662885.2109208 ----------------------
Jobs: 9/9 (100.0%) | Time: 3.64s | Rate: 2.47/s
Memory usage: 165.59 MB (61.50%) | CPU usage: 0.00%



In [11]:
load_info = pipeline.run(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 164.41 MB (61.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.30s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 13981013.33/s
Memory usage: 164.41 MB (61.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.77s | Rate: 0.00/s
books: 200  | Time: 1.47s | Rate: 135.86/s
Memory usage: 164.41 MB (61.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.90s | Rate: 0.00/s
books: 400  | Time: 2.60s | Rate: 153.94/s
Memory usage: 164.41 MB (61.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 5.10s | Rate: 0

In [12]:
ds = pipeline.dataset()

In [13]:
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [18]:
df = ds.books.df()      # main table
df.head(5)

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,396,1997,True,/works/OL82563W,OL38565767M,harrypotterylapi0000rowl_q5r6,False,Harry Potter and the Philosopher's Stone,1771662988.8568764,jsAiImM12QbYmQ,NaN
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,NaN,NaN,False,Harry Potter and the Deathly Hallows,1771662988.8568764,GLBgopbzRvPv4Q,NaN
2,OL26234270M,10580435,borrowable,278,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1771662988.8568764,mQYfpKnabqaGOA,NaN
3,OL25683482M,12059372,printdisabled,242,2000,True,/works/OL82560W,NaN,NaN,False,Harry Potter and the Goblet of Fire,1771662988.8568764,x8H7WfJ4zwM+jQ,NaN
4,OL25662116M,15158666,printdisabled,246,2003,True,/works/OL82548W,NaN,NaN,False,Harry Potter and the Order of the Phoenix,1771662988.8568764,B8IOQVTmFRJgFQ,NaN
